In [ ]:
# %% Cell 1 — imports
import numpy as np
import os
from numba import njit
from tqdm import tqdm
import time

# %% Cell 2 — Swendsen-Wang step (triangular lattice)
@njit
def fast_swendsen_wang_step_tri(spins, L, p, h_bonds, v_bonds, d_bonds, cluster_labels):
    for i in range(L):
        for j in range(L):
            h_bonds[i, j] = False
            v_bonds[i, j] = False
            d_bonds[i, j] = False
            cluster_labels[i, j] = 0

    # 1. Create bonds (Right, Down, Down-Right)
    for i in range(L):
        for j in range(L):
            r_i, r_j = i, (j + 1) % L
            if spins[i, j] == spins[r_i, r_j] and np.random.rand() < p:
                h_bonds[i, j] = True

            d_i, d_j = (i + 1) % L, j
            if spins[i, j] == spins[d_i, d_j] and np.random.rand() < p:
                v_bonds[i, j] = True

            diag_i, diag_j = (i + 1) % L, (j + 1) % L
            if spins[i, j] == spins[diag_i, diag_j] and np.random.rand() < p:
                d_bonds[i, j] = True

    current_label = 1
    stack_i = np.empty(L * L, dtype=np.int32)
    stack_j = np.empty(L * L, dtype=np.int32)

    # 2. Cluster building and flipping
    for i in range(L):
        for j in range(L):
            if cluster_labels[i, j] == 0:
                new_spin = 1 if np.random.rand() < 0.5 else -1

                cluster_labels[i, j] = current_label
                spins[i, j] = new_spin

                stack_i[0] = i
                stack_j[0] = j
                stack_ptr = 1

                while stack_ptr > 0:
                    stack_ptr -= 1
                    curr_i = stack_i[stack_ptr]
                    curr_j = stack_j[stack_ptr]

                    # Right
                    right_j = (curr_j + 1) % L
                    if h_bonds[curr_i, curr_j] and cluster_labels[curr_i, right_j] == 0:
                        cluster_labels[curr_i, right_j] = current_label
                        spins[curr_i, right_j] = new_spin
                        stack_i[stack_ptr] = curr_i
                        stack_j[stack_ptr] = right_j
                        stack_ptr += 1

                    # Left
                    left_j = (curr_j - 1) % L
                    if h_bonds[curr_i, left_j] and cluster_labels[curr_i, left_j] == 0:
                        cluster_labels[curr_i, left_j] = current_label
                        spins[curr_i, left_j] = new_spin
                        stack_i[stack_ptr] = curr_i
                        stack_j[stack_ptr] = left_j
                        stack_ptr += 1

                    # Down
                    down_i = (curr_i + 1) % L
                    if v_bonds[curr_i, curr_j] and cluster_labels[down_i, curr_j] == 0:
                        cluster_labels[down_i, curr_j] = current_label
                        spins[down_i, curr_j] = new_spin
                        stack_i[stack_ptr] = down_i
                        stack_j[stack_ptr] = curr_j
                        stack_ptr += 1

                    # Up
                    up_i = (curr_i - 1) % L
                    if v_bonds[up_i, curr_j] and cluster_labels[up_i, curr_j] == 0:
                        cluster_labels[up_i, curr_j] = current_label
                        spins[up_i, curr_j] = new_spin
                        stack_i[stack_ptr] = up_i
                        stack_j[stack_ptr] = curr_j
                        stack_ptr += 1

                    # Down-Right
                    dr_i = (curr_i + 1) % L
                    dr_j = (curr_j + 1) % L
                    if d_bonds[curr_i, curr_j] and cluster_labels[dr_i, dr_j] == 0:
                        cluster_labels[dr_i, dr_j] = current_label
                        spins[dr_i, dr_j] = new_spin
                        stack_i[stack_ptr] = dr_i
                        stack_j[stack_ptr] = dr_j
                        stack_ptr += 1

                    # Up-Left
                    ul_i = (curr_i - 1) % L
                    ul_j = (curr_j - 1) % L
                    if d_bonds[ul_i, ul_j] and cluster_labels[ul_i, ul_j] == 0:
                        cluster_labels[ul_i, ul_j] = current_label
                        spins[ul_i, ul_j] = new_spin
                        stack_i[stack_ptr] = ul_i
                        stack_j[stack_ptr] = ul_j
                        stack_ptr += 1

                current_label += 1

    return L * L


class IsingMC_SwendsenWang_Triangular:
    def __init__(self, length, temperature=0.):
        self.spins = np.ones((length, length), dtype=np.int32)
        self.L = length
        self.T = temperature
        self.M = length * length
        self.sw_prob = 0.0

        self.cluster_labels = np.zeros((length, length), dtype=np.int32)
        self.h_bonds = np.zeros((length, length), dtype=np.bool_)
        self.v_bonds = np.zeros((length, length), dtype=np.bool_)
        self.d_bonds = np.zeros((length, length), dtype=np.bool_)

        self.update_probabilities()

    def update_probabilities(self):
        if self.T > 0.:
            self.sw_prob = 1.0 - np.exp(-2.0 / self.T)
        else:
            self.sw_prob = 0.0

    def set_temperature(self, temperature):
        self.T = temperature
        self.update_probabilities()

    def reset_spins(self):
        self.spins.fill(1)
        self.M = self.L * self.L

    def swendsen_wang_step(self):
        return fast_swendsen_wang_step_tri(
            self.spins, self.L, self.sw_prob,
            self.h_bonds, self.v_bonds, self.d_bonds, self.cluster_labels
        )

# %% Cell 3 — parameters
J = 1
temps = np.linspace(2.0, 5.0, 40)
L_values = [10, 20, 30, 40, 60]

Nthermalization = 100000
Nsample = 250  # uncorrelated configurations per temperature

output_dir = 'tri_data_centered'
os.makedirs(output_dir, exist_ok=True)

# %% Cell 4 — generate and save
def simulate_temperature(sim, T, L, N, Nthermalization, Nsample, Nsubsweep):
    sim.reset_spins()
    sim.set_temperature(T)

    # Thermalization
    for _ in range(Nthermalization):
        spins_flipped = 0
        while spins_flipped < N:
            spins_flipped += sim.swendsen_wang_step()

    configs = np.zeros((Nsample, N), dtype=np.int8)
    configs[0] = sim.spins.flatten()

    for n in range(1, Nsample):
        spins_flipped = 0
        while spins_flipped < Nsubsweep:
            spins_flipped += sim.swendsen_wang_step()
        configs[n] = sim.spins.flatten()

    return configs


for L in L_values:
    t_start = time.perf_counter()
    N = L ** 2
    Nsubsweep = N * 500

    sim = IsingMC_SwendsenWang_Triangular(L)

    temperatures_arr = np.zeros(len(temps) * Nsample, dtype=np.float32)
    spins_arr        = np.zeros((len(temps) * Nsample, N), dtype=np.int8)

    for t_idx, T in enumerate(tqdm(temps, desc=f"L={L}", leave=False)):
        configs = simulate_temperature(sim, T, L, N, Nthermalization, Nsample, Nsubsweep)
        start = t_idx * Nsample
        end   = start + Nsample
        temperatures_arr[start:end] = T
        spins_arr[start:end] = configs

    filename = os.path.join(output_dir, f'L{L}_tri.npz')
    np.savez_compressed(filename, temperatures=temperatures_arr, spins=spins_arr)
    print(f"Saved L={L} to {filename}   ({time.perf_counter()-t_start:.1f}s)")

Saved L=10 to tri_data_centered/L10_tri.npz   (69.5s)


Saved L=20 to tri_data_centered/L20_tri.npz   (257.9s)


Saved L=30 to tri_data_centered/L30_tri.npz   (583.9s)


Saved L=40 to tri_data_centered/L40_tri.npz   (1094.8s)


L=60:  28%|██▊       | 11/40 [15:05<44:00, 91.06s/it] 